In [38]:
%pip install supabase

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import sys 
from pathlib import Path

project_root = None
for p in Path.cwd().resolve().parents:
    if (p / "utils").exists() and (p / "data").exists():
        project_root = p
        break

if project_root is None:
    raise RuntimeError("Raíz no encontrada.")

sys.path.insert(0, str(project_root))

In [43]:
import os
import logging
import requests
import pandas as pd
from math import ceil
from dotenv import load_dotenv
from utils.paths import find_project_root
from src.transformation import (
    drop_columns,
    dedupe_jobs,
    normalize_string_column,
    rename_columns,
    change_values,
    convert_timestamps,
    add_missing_flags
)


logging.basicConfig(
	level=logging.INFO,
	format="%(asctime)s | %(levelname)s | %(message)s"
)

load_dotenv()

url = os.getenv("SUPABASE_URI")
key = os.getenv("SUPABASE_SERVICE_KEY") 

TABLE = "jobs"

root = find_project_root()
csv_processed_path = root / "data" / "processed" / "job_market_processed.csv"

headers = {
    "apikey": key,
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json",
    "Prefer": "return=minimal"    
}

df_proccesed = drop_columns(df)
df_proccesed = dedupe_jobs(df_proccesed)
for col in columns_to_normalize:
	df_proccesed = normalize_string_column(df_proccesed, col)
df_proccesed = rename_columns(df_proccesed)
df_proccesed = change_values(df_proccesed)
df_proccesed = convert_timestamps(df_proccesed)
df_proccesed = add_missing_flags(df_proccesed, ["company", "company_url", "location", "max_amount", "min_amount", "listing"])  
df_proccesed = df_proccesed.where(pd.notnull(df_proccesed), None)
df_proccesed = df_proccesed.loc[:, ~df_proccesed.columns.duplicated()]

df_proccesed.columns = (
    df_proccesed.columns
        .str.replace(r"\.\d+$", "", regex=True)
        .str.replace("_x$", "", regex=True)
        .str.replace("_y$", "", regex=True)
)


records = df_proccesed.to_dict(orient="records")

batch = 500
total_batches = ceil(len(records) / batch)

for i in range(total_batches):
    chunk = records[i*batch:(i+1)*batch]
    logging.info(f"Subiendo Batch {i+1}/{total_batches} ({len(chunk)} filas)")
    res = requests.post(
        f"{url}/rest/v1/{TABLE}",
        headers=headers,
        json=chunk
    )
    
    if res.status_code not in [200, 201, 204]:
        logging.error("Error en el batch", res.text)
        break
    
logging.info("Carga completa.")

C:\Users\gianlu\AppData\Local\Temp\ipykernel_16924\3495177608.py:60: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  records = df_proccesed.to_dict(orient="records")
2025-12-02 12:27:25,398 | INFO | Subiendo Batch 1/32 (500 filas)
2025-12-02 12:27:26,937 | INFO | Subiendo Batch 2/32 (500 filas)
2025-12-02 12:27:28,053 | INFO | Subiendo Batch 3/32 (500 filas)
2025-12-02 12:27:28,660 | INFO | Subiendo Batch 4/32 (500 filas)
2025-12-02 12:27:29,290 | INFO | Subiendo Batch 5/32 (500 filas)
2025-12-02 12:27:30,992 | INFO | Subiendo Batch 6/32 (500 filas)
2025-12-02 12:27:32,297 | INFO | Subiendo Batch 7/32 (500 filas)
2025-12-02 12:27:33,405 | INFO | Subiendo Batch 8/32 (500 filas)
2025-12-02 12:27:34,102 | INFO | Subiendo Batch 9/32 (500 filas)
2025-12-02 12:27:34,974 | INFO | Subiendo Batch 10/32 (500 filas)
2025-12-02 12:27:35,604 | INFO | Subiendo Batch 11/32 (500 filas)
2025-12-02 12:27:36,050 | INFO | Subiendo Batch 12/32 (500 filas)
2025-12-02 12:27:36,